# KAT-VAD v2 — P4 / **E2(d)**: VideoMAE V2-B vs V2-S on **DoTA-CAP-dev** (no training)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§12 (N1–N12)** on the endpoint of **Amendment 4 (D13)**.
Plan: `.project/plans/katvad-v2-e0-e2.md` P4. One decision comes out: **PICK B / PICK S / MOTION_DROPPED**.

| step | what | GPU? |
|---|---|---|
| 0 | setup, deps | — |
| 1 | preflight: §12 in the tree, DoTA-CAP splits frozen, caches + manifests present (N12) | no |
| 2 | **VideoMAE V2-S on the 300 K sources** (N4) — same geometry as K's V2-B; resumable, sharded 7z → farm → encode → delete | **yes** |
| 3 | D15 (ii): A0 = E1's three KIP-off checkpoints at protocol B on `dota_dev`, per-clip AUC (`protocol_b_eval`) | yes (or slow CPU) |
| 4 | stage the probe inputs to VM disk, run `e2d_probe` (the rule is applied by the tool) | no |
| 5 | record | — |

**Do not change a flag after reading §4.** Every rule was committed before the first number (`7b1889a`).
Name every DoTA-CAP number **DoTA-CAP (n/1397)** and never put it beside a full-DoTA number or 62.60 (D14).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # data/, cache/ (VideoMAE + CLIP caches), v1 outputs
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # kat-vad/ (code), outputs/ (v2 read-outs)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V2_E2D_TRANSFER_MIN'), 'this upload predates §12 -- re-upload branch v2'

ENC_B, ENC_S = constants.VIDEOMAE_ENCODER_B, constants.VIDEOMAE_ENCODER_S
DATASET  = constants.DADA_ORIGIN_DATASET                    # 'DADA2000_orig'
STRIDE   = constants.FRAME_STRIDE                           # 8: K's geometry
T2       = constants.DATA_ROOT / DATASET
CENSUS   = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
T2_CLIP  = constants.CLIP_CACHE_DIR / DATASET
VIDEO    = constants.VIDEO_CACHE_DIR
DOTA     = constants.DATA_ROOT / 'DoTA'
LABELS   = DOTA / 'labels_s8'                               # defs.json only
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
# K's report was written under the old layout (Thesis/outputs); accept either.
K_DIRS   = [constants.OUTPUT_ROOT / 'REPORTS' / 'v2_K', V1_OUTPUTS / 'REPORTS' / 'v2_K']
K_REPORT = next((d for d in K_DIRS if (d / 'k_sources.txt').is_file()), None)
assert K_REPORT is not None, f'no k_sources.txt under {K_DIRS}'
REPORT   = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_E2d'
A0_DIR   = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_A0_dev_protocolB'
LOCAL    = Path('/content/e2d')                              # VM-local NVMe, never Drive
CKPTS = {                                                    # E1's checkpoints (D12)
    's2024': V1_OUTPUTS / 'DADA2000_orig_phase4' / 's2024' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2025': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2025' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2026': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2026' / 'stage2_kip_off' / 'checkpoint_last.pt',
}
for p in (REPORT, A0_DIR, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name in ('T2', 'CENSUS', 'ORIG', 'T2_CLIP', 'DOTA_S1', 'K_REPORT', 'REPORT'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
apt-get -qq install -y p7zip-full
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time

ENV = dict(os.environ)


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'w') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')

## 1. Preflight (N12) — refuse to start on a tree, split or cache that is not the registered one

In [ ]:
from core.data.v2_splits import lines_sha1, load_split
from core.tools import e2d_probe, dota_cap, extract_video_features

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 12. E2(d) on DoTA-CAP' in addendum, 'this upload predates addendum §12'
cap_dev = load_split(constants.V2_SPLIT_DOTA_CAP_DEV)     # sha1-checked
dev = load_split(constants.V2_SPLIT_DOTA_DEV)
assert len(cap_dev) == 569 and set(cap_dev) <= set(dev), (len(cap_dev), 'DoTA-CAP split is not the frozen one')

k_ids = [l.strip() for l in (K_REPORT / 'k_sources.txt').read_text().splitlines() if l.strip()]
k_sha1 = json.loads((K_REPORT / 'k_sources_manifest.json').read_text())['sha1']
assert lines_sha1(k_ids) == k_sha1, 'K sources do not match their manifest'
census = json.loads(CENSUS.read_text())
assert all(v in census for v in k_ids), 'a K source has no census entry'

alignment = e2d_probe.dota_cap_alignment_sha256(constants.V2_SPLITS_DIR)
for enc in (ENC_B, ENC_S):
    d = e2d_probe.dota_video_dir(VIDEO, enc)
    e2d_probe.check_manifest(d, dota_cap.video_manifest(enc, alignment))
    miss = [v for v in cap_dev if not (d / f'{v}.npy').exists()]
    assert not miss, f'{enc}: {len(miss)} DoTA-CAP-dev clips have no feature: {miss[:3]}'
b_dir = e2d_probe.t2_video_dir(VIDEO, ENC_B)
e2d_probe.check_manifest(b_dir, extract_video_features.build_manifest(
    ENC_B, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC_B]))
assert all((b_dir / f'{v}.npy').exists() for v in k_ids), 'K V2-B cache incomplete'
assert all((T2_CLIP / f'{v}.npy').exists() for v in k_ids), 'K CLIP cache incomplete'
assert all((DOTA_S1 / f'{v}.npy').exists() for v in dev), 'DoTA_s1_ncc incomplete on dota_dev'
print(f'N12 OK: dota_cap_dev {len(cap_dev)} | dota_dev {len(dev)} | K {len(k_ids)} sources (sha1 {k_sha1[:12]}) '
      f'| alignment {alignment[:12]}')

## 2. VideoMAE **V2-S** on the 300 K sources (N4) — GPU, resumable

Same pipeline and geometry as K's V2-B (`p1_kill_switch.ipynb` §3): 16 frames, every 3rd at the assumed 30 fps
(1.5 s), causal, squash 224², stride 8. The extractor writes `video_manifest.json` and refuses to extend a cache
built with other settings (C2). A shard whose ids are all encoded is skipped.

In [ ]:
S_DIR = e2d_probe.t2_video_dir(VIDEO, ENC_S)
SHARD, BATCH = 60, 32
root, images = constants.DADA_ORIGIN_ROOT_DIRNAME, constants.DADA_ORIGIN_IMAGES_SUBDIR
assert (ORIG / 'DADA2000.zip').is_file(), f'no DADA2000.zip under {ORIG}'


def type_video(vid):
    t, v = vid.split('_')                      # t{type:02d}_v{video:03d}
    return int(t[1:]), int(v[1:])


started = time.monotonic()
for k in range(0, len(k_ids), SHARD):
    shard, tag = k_ids[k:k + SHARD], f'{k // SHARD:03d}'
    if all((S_DIR / f'{v}.npy').exists() for v in shard):
        print(f'shard {tag}: already encoded, skipping')
        continue
    frames, farm = LOCAL / 'frames', LOCAL / 'flat'
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    farm.mkdir(parents=True)
    patterns = [f'{root}/{t}/{v:03d}/{images}/*' for t, v in map(type_video, shard)]
    print(f'--- shard {tag}: {len(shard)} sources, {shutil.disk_usage("/content").free / 2**30:.1f} GiB free')
    run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', *patterns])
    for vid in shard:                          # C26: link each source's images dir under its unique id
        t, v = type_video(vid)
        src = frames / root / str(t) / f'{v:03d}' / images
        n = len(list(src.glob('*')))
        if n != census[vid]:                   # C10: a folder is not evidence of data
            raise RuntimeError(f'{vid}: {n} images extracted, census says {census[vid]}')
        (farm / vid).symlink_to(src.resolve(), target_is_directory=True)
    shard_ids = LOCAL / f'ids_{tag}.txt'
    shard_ids.write_text('\n'.join(shard) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_video_features',
         '--frames-dir', farm, '--ids-file', shard_ids, '--dataset', DATASET,
         '--encoder', ENC_S, '--stride', STRIDE, '--batch-size', BATCH,
         '--device', 'auto', '--output-dir', S_DIR])
    shutil.rmtree(frames, ignore_errors=True)
    shutil.rmtree(farm, ignore_errors=True)
    print(f'    encoded {sum((S_DIR / f"{v}.npy").exists() for v in k_ids)}/{len(k_ids)} | '
          f'{(time.monotonic() - started) / 60:.1f} min')

assert all((S_DIR / f'{v}.npy').exists() for v in k_ids), 'K V2-S cache incomplete -- rerun this cell'
e2d_probe.check_manifest(S_DIR, extract_video_features.build_manifest(
    ENC_S, STRIDE, constants.VIDEOMAE_WEIGHTS_SHA256[ENC_S]))
print('V2-S on K sources: complete, manifest matches V2-B geometry')

## 3. D15 (ii) — A0 at protocol B on `dota_dev`, per clip (printed, never decided on)

E1's three finished KIP-off checkpoints (J10 is re-checked by the tool), plain `DoTA_s1_ncc`, stride 3. Writes
`clip_aucs.json`, which §4 splits into all / DoTA-CAP / dropped. Skipped if it already exists.

In [ ]:
from core.tools import protocol_b_eval

if (A0_DIR / protocol_b_eval.CLIP_AUCS_JSON).exists():
    print(f'reusing {A0_DIR / protocol_b_eval.CLIP_AUCS_JSON}')
else:
    local_s1 = LOCAL / 'DoTA_s1_ncc_dev'
    local_s1.mkdir(parents=True, exist_ok=True)
    for v in dev:
        if not (local_s1 / f'{v}.npy').exists():
            shutil.copy2(DOTA_S1 / f'{v}.npy', local_s1 / f'{v}.npy')
    run_args = []
    for name, ckpt in CKPTS.items():
        assert ckpt.is_file(), f'missing {ckpt}'
        run_args += ['--run', name, ckpt]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *run_args,
         '--input-dir', local_s1, '--s1-dir', local_s1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', LABELS, '--split', constants.V2_SPLIT_DOTA_DEV,
         '--out-dir', A0_DIR, '--device', 'auto'], log=A0_DIR / 'run.log')
print((A0_DIR / protocol_b_eval.READOUT_MD).read_text())

## 4. **E2(d)** — probes, intervals, decision (N1–N9), D15 printed

Inputs are staged to VM disk first (thousands of small `.npy` from Drive are slow). The staged tree keeps the
cache layout (`<root>/<encoder>/<dataset>_s*_squash/` + `video_manifest.json`), so the tool's N12 gates re-run on it.
Expect ~30–60 min of CPU (two arms × two encoders × two probes × six feature sets, 5 folds).

In [ ]:
stage = LOCAL / 'stage'
v_root = stage / 'video'
jobs = [(DOTA_S1, stage / 'DoTA_s1_ncc', dev), (T2_CLIP, stage / 'clip_t2', k_ids)]
for enc in (ENC_B, ENC_S):
    for src_dir, ids in ((e2d_probe.dota_video_dir(VIDEO, enc), cap_dev),
                         (e2d_probe.t2_video_dir(VIDEO, enc), k_ids)):
        jobs.append((src_dir, v_root / enc / src_dir.name, ids))
t0 = time.time()
for src_dir, dst_dir, ids in jobs:
    dst_dir.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst_dir / f'{v}.npy').exists():
            shutil.copy2(src_dir / f'{v}.npy', dst_dir / f'{v}.npy')
    manifest = src_dir / constants.VIDEO_MANIFEST_FILENAME
    if manifest.exists():
        shutil.copy2(manifest, dst_dir / manifest.name)
print(f'staged in {time.time() - t0:.0f}s')

a0_aucs = A0_DIR / protocol_b_eval.CLIP_AUCS_JSON
run([sys.executable, '-m', 'core.tools.e2d_probe',
     '--dota-s1-dir', stage / 'DoTA_s1_ncc', '--metadata', DOTA / 'metadata_val.json',
     '--split-file', DOTA / 'val_split.txt',
     '--k-ids-file', K_REPORT / 'k_sources.txt', '--k-manifest', K_REPORT / 'k_sources_manifest.json',
     '--annotation', ORIG / constants.DADA_ORIGIN_ANNOTATION_FILENAME, '--census', CENSUS,
     '--t2-clip-dir', stage / 'clip_t2', '--video-root', v_root,
     *(['--a0-clip-aucs', a0_aucs] if a0_aucs.exists() else []),
     '--out-dir', REPORT], log=REPORT / 'e2d_run.log')
print((REPORT / e2d_probe.READOUT_MD).read_text())

## 5. Record — then **stop**

Paste `e2d_readout.md` (and `v2_A0_dev_protocolB/protocol_b_readout.md`) back to Claude. The decision is the tool's:

* **PICK B / PICK S** → plan P4 last step: VideoMAE on **full T2** for that encoder only, then bake A2/A3 (pilot batch 2).
* **MOTION_DROPPED** → A2/A3 leave E3; the pilot is batch 1 (A0/A1) only (D16).

No flag is changed after reading the numbers (lesson 14).